# Nettoyage et préparation des données

Ce notebook applique les décisions prises pendant la compréhension des données.

Les objectifs sont :

- charger les données brutes ;
- supprimer les colonnes inutiles ;
- contrôler les doublons ;
- vérifier les types et les valeurs ;
- créer les variables cibles ;
- enregistrer un dataset nettoyé et reproductible.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
raw_data_path = Path("../data/raw/data_ex.csv")
processed_data_path = Path("../data/processed/cleaned_data.csv")

In [4]:
df_raw = pd.read_csv(raw_data_path)

df_raw.head()

,Unnamed: 0,gender,Age_client,year,age_of_car_M,Car_power_M,Car_2ndDriver_M,num_policiesC,metro_code,Policy_PaymentMethodA,...,Insuredcapital_continent_re,appartment,Client_Seniority,Retention,NClaims1,NClaims2,Claims1,Claims2,Types,PolID
0,1,1,84,1,13,90.0,0,1,0,0,...,11.549088,0,16.963723,1,0,0,0.0,0.0,1,1
1,2,1,83,1,0,177.0,0,1,0,1,...,11.506740,0,16.796715,1,0,0,0.0,0.0,1,2
2,3,1,85,1,0,163.0,0,1,0,1,...,11.575080,1,18.603696,1,0,0,0.0,0.0,1,3
3,4,1,85,1,0,90.0,0,1,0,1,...,12.156664,1,20.484600,1,0,0,0.0,0.0,1,4
4,5,1,82,1,20,115.0,0,1,0,1,...,12.369568,1,7.901437,1,0,0,0.0,0.0,1,5


In [5]:
print(f"Dimensions des données brutes : {df_raw.shape}")

Dimensions des données brutes : (122935, 22)


In [6]:
df_clean = df_raw.copy()

In [7]:
print(f"Dimensions de df_raw   : {df_raw.shape}")
print(f"Dimensions de df_clean : {df_clean.shape}")

Dimensions de df_raw   : (122935, 22)
Dimensions de df_clean : (122935, 22)


In [8]:
"Unnamed: 0" in df_clean.columns

True

In [9]:
df_clean = df_clean.drop(columns=["Unnamed: 0"])

In [10]:
print(f"Dimensions après suppression : {df_clean.shape}")
print(df_clean.columns.tolist())

Dimensions après suppression : (122935, 21)
['gender', 'Age_client', 'year', 'age_of_car_M', 'Car_power_M', 'Car_2ndDriver_M', 'num_policiesC', 'metro_code', 'Policy_PaymentMethodA', 'Policy_PaymentMethodH', 'Insuredcapital_content_re', 'Insuredcapital_continent_re', 'appartment', 'Client_Seniority', 'Retention', 'NClaims1', 'NClaims2', 'Claims1', 'Claims2', 'Types', 'PolID']


In [11]:
df_clean = df_clean.drop(
    columns=["Unnamed: 0"],
    errors="ignore"
)

In [12]:
duplicate_count = df_clean.duplicated().sum()

print(f"Nombre de lignes dupliquées : {duplicate_count}")

Nombre de lignes dupliquées : 0


In [13]:
rows_before = len(df_clean)

df_clean = df_clean.drop_duplicates()

rows_after = len(df_clean)

print(f"Lignes avant : {rows_before}")
print(f"Lignes après : {rows_after}")
print(f"Lignes supprimées : {rows_before - rows_after}")

Lignes avant : 122935
Lignes après : 122935
Lignes supprimées : 0


In [14]:
policy_year_duplicates = df_clean.duplicated(
    subset=["PolID", "year"]
).sum()

print(
    "Doublons de la combinaison PolID-year : "
    f"{policy_year_duplicates}"
)

Doublons de la combinaison PolID-year : 0


## Résultat du nettoyage structurel

- La colonne `Unnamed: 0`, correspondant à un ancien index, a été supprimée.
- Aucune ligne complètement dupliquée n’a été détectée.
- La combinaison `PolID`–`year` reste unique.
- Le dataset nettoyé contient actuellement 122 935 lignes et 21 colonnes.

## 2. Validation des types et des valeurs

Cette section vérifie que les variables possèdent les types et les modalités attendus avant la création des variables cibles.

In [15]:
df_clean.dtypes

gender                           int64
Age_client                       int64
year                             int64
age_of_car_M                     int64
Car_power_M                    float64
Car_2ndDriver_M                  int64
num_policiesC                    int64
metro_code                       int64
Policy_PaymentMethodA            int64
Policy_PaymentMethodH            int64
Insuredcapital_content_re      float64
Insuredcapital_continent_re    float64
appartment                       int64
Client_Seniority               float64
Retention                        int64
NClaims1                         int64
NClaims2                         int64
Claims1                        float64
Claims2                        float64
Types                            int64
PolID                            int64
dtype: object

In [16]:
type_summary = pd.DataFrame({
    "Type": df_clean.dtypes,
    "Valeurs_uniques": df_clean.nunique(),
    "Valeurs_manquantes": df_clean.isnull().sum()
})

type_summary

,Type,Valeurs_uniques,Valeurs_manquantes
gender,int64,2,0
Age_client,int64,75,0
year,int64,5,0
age_of_car_M,int64,40,0
Car_power_M,float64,281,0
Car_2ndDriver_M,int64,2,0
num_policiesC,int64,2,0
metro_code,int64,2,0
Policy_PaymentMethodA,int64,2,0
Policy_PaymentMethodH,int64,2,0


In [17]:
identifier_columns = [
    "PolID"
]

temporal_columns = [
    "year"
]

binary_columns = [
    "gender",
    "Car_2ndDriver_M",
    "num_policiesC",
    "metro_code",
    "Policy_PaymentMethodA",
    "Policy_PaymentMethodH",
    "appartment",
    "Retention"
]

numerical_columns = [
    "Age_client",
    "age_of_car_M",
    "Car_power_M",
    "Insuredcapital_content_re",
    "Insuredcapital_continent_re",
    "Client_Seniority"
]

claim_count_columns = [
    "NClaims1",
    "NClaims2"
]

claim_amount_columns = [
    "Claims1",
    "Claims2"
]

In [18]:
for column in binary_columns:
    values = sorted(df_clean[column].unique())

    print(f"{column} : {values}")

gender : [np.int64(0), np.int64(1)]
Car_2ndDriver_M : [np.int64(0), np.int64(1)]
num_policiesC : [np.int64(0), np.int64(1)]
metro_code : [np.int64(0), np.int64(1)]
Policy_PaymentMethodA : [np.int64(0), np.int64(1)]
Policy_PaymentMethodH : [np.int64(0), np.int64(1)]
appartment : [np.int64(0), np.int64(1)]
Retention : [np.int64(0), np.int64(1)]


In [19]:
binary_validation = {}

for column in binary_columns:
    valid_values = df_clean[column].isin([0, 1])
    invalid_count = (~valid_values).sum()

    binary_validation[column] = invalid_count

binary_validation

{'gender': np.int64(0),
 'Car_2ndDriver_M': np.int64(0),
 'num_policiesC': np.int64(0),
 'metro_code': np.int64(0),
 'Policy_PaymentMethodA': np.int64(0),
 'Policy_PaymentMethodH': np.int64(0),
 'appartment': np.int64(0),
 'Retention': np.int64(0)}

In [20]:
for column in binary_columns:
    assert df_clean[column].isin([0, 1]).all(), (
        f"Valeur invalide détectée dans {column}"
    )

In [21]:
print("Toutes les variables binaires contiennent uniquement 0 et 1.")

Toutes les variables binaires contiennent uniquement 0 et 1.


In [22]:
non_negative_columns = [
    "Age_client",
    "age_of_car_M",
    "Car_power_M",
    "Insuredcapital_content_re",
    "Insuredcapital_continent_re",
    "Client_Seniority",
    "NClaims1",
    "NClaims2",
    "Claims1",
    "Claims2"
]

In [23]:
negative_values = {}

for column in non_negative_columns:
    negative_count = (df_clean[column] < 0).sum()
    negative_values[column] = negative_count

negative_values

{'Age_client': np.int64(0),
 'age_of_car_M': np.int64(0),
 'Car_power_M': np.int64(0),
 'Insuredcapital_content_re': np.int64(0),
 'Insuredcapital_continent_re': np.int64(0),
 'Client_Seniority': np.int64(0),
 'NClaims1': np.int64(0),
 'NClaims2': np.int64(0),
 'Claims1': np.int64(0),
 'Claims2': np.int64(0)}

In [24]:
for column in non_negative_columns:
    assert (df_clean[column] >= 0).all(), (
        f"Valeur négative détectée dans {column}"
    )

print("Aucune valeur négative détectée.")

Aucune valeur négative détectée.


In [25]:
for column in claim_count_columns:
    is_integer = (
        df_clean[column]
        == df_clean[column].astype(int)
    ).all()

    print(f"{column} contient uniquement des entiers : {is_integer}")

NClaims1 contient uniquement des entiers : True
NClaims2 contient uniquement des entiers : True


In [26]:
df_clean[claim_count_columns] = (
    df_clean[claim_count_columns].astype("int64")
)

In [27]:
df_clean["Age_client"].agg(["min", "max"])

min    18
max    95
Name: Age_client, dtype: int64

In [28]:
valid_age = df_clean["Age_client"].between(18, 100)

print(f"Âges hors intervalle : {(~valid_age).sum()}")

Âges hors intervalle : 0


In [29]:
assert valid_age.all(), "Certains âges sont hors de l'intervalle 18–100."

print("Tous les âges sont compris entre 18 et 100 ans.")

Tous les âges sont compris entre 18 et 100 ans.


In [30]:
sorted(df_clean["year"].unique())

[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

In [31]:
expected_years = {1, 2, 3, 4, 5}
observed_years = set(df_clean["year"].unique())

assert observed_years == expected_years, (
    f"Années inattendues : {observed_years}"
)

print("Les années disponibles sont bien 1, 2, 3, 4 et 5.")

Les années disponibles sont bien 1, 2, 3, 4 et 5.


### Décision concernant les valeurs extrêmes

Les contrôles ne montrent aucune valeur négative ni aucun âge manifestement impossible.

Certaines valeurs, comme un véhicule âgé de 68 ans, une puissance de 560 ou des montants de sinistres très élevés, sont statistiquement atypiques. Elles ne sont cependant pas supprimées automatiquement.

Dans ce projet, les sinistres extrêmes contiennent une information importante sur le risque de queue. Leur influence sera traitée pendant le feature engineering et la modélisation, notamment avec des transformations logarithmiques et des modèles adaptés.

## 3. Création des variables cibles

Les variables de sinistres de type 1 et de type 2 sont regroupées afin de construire les cibles nécessaires à la classification, à la modélisation de la fréquence et à la modélisation du coût.

In [32]:
df_clean["Total_NClaims"] = (
    df_clean["NClaims1"]
    + df_clean["NClaims2"]
)

In [33]:
df_clean[
    ["NClaims1", "NClaims2", "Total_NClaims"]
].head(10)

,NClaims1,NClaims2,Total_NClaims
0,0,0,0
1,0,0,0
2,0,0,0
3,0,0,0
4,0,0,0
5,0,0,0
6,0,0,0
7,0,0,0
8,0,0,0
9,0,0,0


In [34]:
assert (
    df_clean["Total_NClaims"]
    == df_clean["NClaims1"] + df_clean["NClaims2"]
).all()

print("La variable Total_NClaims est correctement calculée.")

La variable Total_NClaims est correctement calculée.


In [35]:
df_clean["Total_Claims"] = (
    df_clean["Claims1"]
    + df_clean["Claims2"]
)

In [36]:
df_clean[
    ["Claims1", "Claims2", "Total_Claims"]
].head(10)

,Claims1,Claims2,Total_Claims
0,0.0,0.0,0.0
1,0.0,0.0,0.0
2,0.0,0.0,0.0
3,0.0,0.0,0.0
4,0.0,0.0,0.0
5,0.0,0.0,0.0
6,0.0,0.0,0.0
7,0.0,0.0,0.0
8,0.0,0.0,0.0
9,0.0,0.0,0.0


In [37]:
assert (
    np.isclose(
        df_clean["Total_Claims"],
        df_clean["Claims1"] + df_clean["Claims2"]
    )
).all()

print("La variable Total_Claims est correctement calculée.")

La variable Total_Claims est correctement calculée.


In [38]:
df_clean["Has_Claim"] = (
    df_clean["Total_NClaims"] > 0
).astype("int64")

In [39]:
df_clean[
    "Has_Claim"
].value_counts(normalize=True).sort_index().mul(100).round(2)

Has_Claim
0    94.43
1     5.57
Name: proportion, dtype: float64

In [40]:
assert df_clean["Has_Claim"].isin([0, 1]).all()

assert (
    df_clean["Has_Claim"]
    == (df_clean["Total_NClaims"] > 0).astype(int)
).all()

print("La variable Has_Claim est correctement calculée.")

La variable Has_Claim est correctement calculée.


In [41]:
df_clean["Has_Paid_Claim"] = (
    df_clean["Total_Claims"] > 0
).astype("int64")

In [42]:
pd.crosstab(
    df_clean["Has_Claim"],
    df_clean["Has_Paid_Claim"],
    margins=True
)

Has_Paid_Claim,0,1,All
Has_Claim,,,
0,116085,0,116085
1,2735,4115,6850
All,118820,4115,122935


In [43]:
paid_without_claim = (
    (df_clean["Has_Claim"] == 0)
    & (df_clean["Has_Paid_Claim"] == 1)
).sum()

print(
    "Paiements positifs sans sinistre déclaré : "
    f"{paid_without_claim}"
)

Paiements positifs sans sinistre déclaré : 0


In [44]:
df_clean["Average_Claim_Severity"] = np.where(
    df_clean["Total_NClaims"] > 0,
    df_clean["Total_Claims"] / df_clean["Total_NClaims"],
    np.nan
)

In [45]:
df_clean.loc[
    df_clean["Has_Claim"] == 1,
    [
        "Total_NClaims",
        "Total_Claims",
        "Average_Claim_Severity"
    ]
].head(10)

,Total_NClaims,Total_Claims,Average_Claim_Severity
14,1,0.00,0.00
23,1,3613.79,3613.79
38,1,0.00,0.00
43,1,882.00,882.00
69,1,288.17,288.17
81,1,0.00,0.00
89,1,882.00,882.00
91,1,105.75,105.75
150,1,0.00,0.00
161,1,85.73,85.73


In [46]:
target_columns = [
    "Total_NClaims",
    "Total_Claims",
    "Has_Claim",
    "Has_Paid_Claim",
    "Average_Claim_Severity"
]

df_clean[target_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
Total_NClaims,122935.0,0.063920,0.281415,0.0,0.0,0.0000,0.0000,6.00
Total_Claims,122935.0,34.347685,586.401931,0.0,0.0,0.0000,0.0000,109720.71
Has_Claim,122935.0,0.055721,0.229382,0.0,0.0,0.0000,0.0000,1.00
Has_Paid_Claim,122935.0,0.033473,0.179869,0.0,0.0,0.0000,0.0000,1.00
Average_Claim_Severity,6850.0,460.993779,1270.548487,0.0,0.0,95.9375,664.9925,36573.57


In [47]:
df_clean[target_columns].isnull().sum()

Total_NClaims                  0
Total_Claims                   0
Has_Claim                      0
Has_Paid_Claim                 0
Average_Claim_Severity    116085
dtype: int64

### Variables cibles créées

- `Has_Claim` : présence d’au moins un sinistre déclaré ;
- `Total_NClaims` : nombre total de sinistres ;
- `Total_Claims` : coût total des sinistres ;
- `Has_Paid_Claim` : présence d’un montant de sinistre positif ;
- `Average_Claim_Severity` : coût moyen par sinistre lorsqu’au moins un sinistre est déclaré.

Les valeurs manquantes de `Average_Claim_Severity` pour les observations sans sinistre sont intentionnelles et ne constituent pas un problème de qualité.

In [48]:
print(f"Dimensions finales : {df_clean.shape}")

Dimensions finales : (122935, 26)


## 4. Séparation des variables explicatives et des cibles

Avant la modélisation, il faut identifier les variables qui ne seraient pas disponibles au moment de réaliser une prédiction.

In [49]:
types_claims_table = pd.crosstab(
    df_clean["Types"],
    [
        df_clean["NClaims1"] > 0,
        df_clean["NClaims2"] > 0
    ]
)

types_claims_table

NClaims1   False       True       
NClaims2   False True  False True 
Types                             
1         116085  1696  1020    19
2              0     0  1899    27
3              0  2123     0    25
4              0     0     0    41

### Risque de fuite associé à `Types`

La variable `Types` est fortement liée à la nature des sinistres observés. Elle semble être déterminée après l’observation des sinistres.

Elle sera conservée dans le dataset nettoyé pour l’analyse, mais exclue des variables explicatives des modèles prédictifs.

In [51]:
identifier_columns = [
    "PolID"
]

target_columns = [
    "NClaims1",
    "NClaims2",
    "Claims1",
    "Claims2",
    "Total_NClaims",
    "Total_Claims",
    "Has_Claim",
    "Has_Paid_Claim",
    "Average_Claim_Severity"
]

leakage_columns = [
    "Types"
]

In [52]:
candidate_feature_columns = [
    "gender",
    "Age_client",
    "age_of_car_M",
    "Car_power_M",
    "Car_2ndDriver_M",
    "num_policiesC",
    "metro_code",
    "Policy_PaymentMethodA",
    "Policy_PaymentMethodH",
    "Insuredcapital_content_re",
    "Insuredcapital_continent_re",
    "appartment",
    "Client_Seniority"
]

In [53]:
temporally_uncertain_columns = [
    "Retention"
]

In [54]:
required_columns = (
    identifier_columns
    + candidate_feature_columns
    + temporally_uncertain_columns
    + target_columns
    + leakage_columns
    + ["year"]
)

missing_required_columns = [
    column
    for column in required_columns
    if column not in df_clean.columns
]

missing_required_columns

[]

In [55]:
assert len(missing_required_columns) == 0, (
    f"Colonnes attendues absentes : {missing_required_columns}"
)

print("Toutes les colonnes attendues sont présentes.")

Toutes les colonnes attendues sont présentes.


In [56]:
variable_roles = {}

for column in df_clean.columns:
    if column in identifier_columns:
        role = "Identifiant"
    elif column == "year":
        role = "Variable temporelle"
    elif column in candidate_feature_columns:
        role = "Variable explicative candidate"
    elif column in temporally_uncertain_columns:
        role = "Disponibilité temporelle à confirmer"
    elif column in target_columns:
        role = "Cible ou variable de sinistre"
    elif column in leakage_columns:
        role = "Variable à risque de fuite"
    else:
        role = "À examiner"

    variable_roles[column] = role

In [57]:
variable_roles_df = pd.DataFrame({
    "Variable": variable_roles.keys(),
    "Role": variable_roles.values()
})

variable_roles_df

,Variable,Role
0,gender,Variable explicative candidate
1,Age_client,Variable explicative candidate
2,year,Variable temporelle
3,age_of_car_M,Variable explicative candidate
4,Car_power_M,Variable explicative candidate
5,Car_2ndDriver_M,Variable explicative candidate
6,num_policiesC,Variable explicative candidate
7,metro_code,Variable explicative candidate
8,Policy_PaymentMethodA,Variable explicative candidate
9,Policy_PaymentMethodH,Variable explicative candidate


In [58]:
print(f"Nombre de lignes : {df_clean.shape[0]}")
print(f"Nombre de colonnes : {df_clean.shape[1]}")
print(f"Doublons : {df_clean.duplicated().sum()}")

print(
    "Valeurs manquantes hors sévérité :",
    df_clean
    .drop(columns=["Average_Claim_Severity"])
    .isnull()
    .sum()
    .sum()
)

Nombre de lignes : 122935
Nombre de colonnes : 26
Doublons : 0
Valeurs manquantes hors sévérité : 0


In [59]:
assert df_clean.shape[0] == df_raw.shape[0]
assert "Unnamed: 0" not in df_clean.columns
assert df_clean.duplicated(["PolID", "year"]).sum() == 0
assert (df_clean["Total_NClaims"] >= 0).all()
assert (df_clean["Total_Claims"] >= 0).all()

print("Tous les contrôles finaux sont validés.")

Tous les contrôles finaux sont validés.


In [60]:
processed_data_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

In [61]:
df_clean.to_csv(
    processed_data_path,
    index=False
)

In [62]:
print(
    f"Dataset nettoyé enregistré dans : "
    f"{processed_data_path}"
)

Dataset nettoyé enregistré dans : ..\data\processed\cleaned_data.csv


In [63]:
df_check = pd.read_csv(processed_data_path)

print(f"Dimensions du fichier rechargé : {df_check.shape}")
df_check.head()

Dimensions du fichier rechargé : (122935, 26)


,gender,Age_client,year,age_of_car_M,Car_power_M,Car_2ndDriver_M,num_policiesC,metro_code,Policy_PaymentMethodA,Policy_PaymentMethodH,...,NClaims2,Claims1,Claims2,Types,PolID,Total_NClaims,Total_Claims,Has_Claim,Has_Paid_Claim,Average_Claim_Severity
0,1,84,1,13,90.0,0,1,0,0,1,...,0,0.0,0.0,1,1,0,0.0,0,0,NaN
1,1,83,1,0,177.0,0,1,0,1,1,...,0,0.0,0.0,1,2,0,0.0,0,0,NaN
2,1,85,1,0,163.0,0,1,0,1,1,...,0,0.0,0.0,1,3,0,0.0,0,0,NaN
3,1,85,1,0,90.0,0,1,0,1,1,...,0,0.0,0.0,1,4,0,0.0,0,0,NaN
4,1,82,1,20,115.0,0,1,0,1,1,...,0,0.0,0.0,1,5,0,0.0,0,0,NaN


In [64]:
assert df_check.shape == df_clean.shape
assert "Unnamed: 0" not in df_check.columns

print("Le fichier nettoyé a été enregistré et rechargé correctement.")

Le fichier nettoyé a été enregistré et rechargé correctement.


## 5. Conclusion

Le dataset brut a été nettoyé sans supprimer d’observations.

La colonne d’index inutile a été retirée, les doublons ont été contrôlés et les domaines de valeurs ont été validés.

Cinq variables liées à la modélisation ont été créées : `Total_NClaims`, `Total_Claims`, `Has_Claim`, `Has_Paid_Claim` et `Average_Claim_Severity`.

La variable `Types` a été identifiée comme une source potentielle de fuite de données et sera exclue des variables explicatives. La disponibilité temporelle de `Retention` devra être confirmée avant son utilisation.

Le fichier nettoyé est enregistré dans `data/processed/cleaned_data.csv`.